In [1]:
import numpy as np
import sys
sys.path.append("../src")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import glob
import os
from utils import time_embedding_np, reward_simulate,simulate_data_raw,make_weighted_target,position_encoder
from utils import best_next_state,path_opt,preward_opt,compute_normalized_future_rewards

In [2]:
base_reward = 10.0
map_name = "short"
decay_rate = 0.6
reward_period = 10 
session_duration = 500000
rewards_in_period = []

total_steps = session_duration
for period_start in range(0, total_steps, reward_period):
    period_end = min(period_start + reward_period, total_steps)
    steps_in_period = np.arange(period_start, period_end)
    rewards_in_period.extend(base_reward * (decay_rate ** (steps_in_period - period_start)))

pattern = [1,1,1,1,1,1,1,2,3,4,5,5,5,5,5,5,5,4,3,2]
optimal_states = np.array(pattern*(session_duration // len(pattern))).reshape(-1,1)


In [4]:
def get_close_colors(cmap_name, start=0.5, delta=0.15,n = 2):
    cmap = cm.get_cmap(cmap_name)
    cmap_color = [cmap(start + i*delta) for i in range(n)]
    return cmap_color[:n]

In [21]:
Ts_DF = [2**i for i in np.arange(7,8)]+[150,200,225,256,300,400,512]+[750]+[1024,1250,1500,2048]
Ts_offline = [2**i for i in np.arange(7,10)]+[750]+[1024,1250,1500,2048]
step = 6
PREWARDS = []
PREGRETS = []
for rep in range(10):
    fn = f"../results/PVAFM/GBT_oh_gamma0.9_weighted_w_time128_2048_lookahead{step}_reps{rep}.npz"
    data = np.load(fn)
    states = data["future_states"]
    prewards = []
    pregrets = []
    for k,T in enumerate(Ts_DF):
        if T in Ts_offline:
            irewards = [reward_simulate(states[k,i],T+i,rewards_in_period) for i in range(1,states.shape[1])]
            preward = compute_normalized_future_rewards(irewards,100,0.5, normalization= True)
            path, ireward_opt = path_opt(states[k,0], T, 100)
            preward_opt = compute_normalized_future_rewards(ireward_opt[1:],100,0.5,normalization= True)
            pregret = (np.sum(preward_opt).item()-np.sum(preward).item())/100
            prewards.append(np.sum(preward).item())
            pregrets.append(pregret)
    PREWARDS.append(prewards)
    PREGRETS.append(pregrets)
    PREWARDS_arr = np.vstack(PREWARDS)
    PREGRETS_rr = np.vstack(PREGRETS)
print(PREGRETS_rr.shape)

T = 128
PREWARDS = []
PREGRETS = []
for rep in range(10):
    fn = f"../results/PVAFM/GBT_oh_gamma0.9_weighted_w_t0_lookahead{step}_reps{rep}.npz"
    data = np.load(fn)
    states = data["future_states"]
    prewards = []
    pregrets = []
    irewards = [reward_simulate(states[0,i],T+i,rewards_in_period) for i in range(1,states.shape[1])]
    preward = compute_normalized_future_rewards(irewards,100,0.5, normalization= True)
    path, ireward_opt = path_opt(states[0,0], T, 100)
    preward_opt = compute_normalized_future_rewards(ireward_opt[1:],100,0.5,normalization= True)
    pregret = (np.sum(preward_opt).item()-np.sum(preward).item())/100
    prewards.append(np.sum(preward).item())
    pregrets.append(pregret)
    PREWARDS.append(prewards)
    PREGRETS.append(pregrets)
    PREWARDS_arr = np.vstack(PREWARDS)
    PREGRETS_init_rr = np.vstack(PREGRETS)
print(PREGRETS_init_rr.shape)

(10, 8)
(10, 1)


In [27]:
Ts_offline = [0] + Ts_offline
print(Ts_offline)
PREGRETS_PVAFM_rr = np.hstack([PREGRETS_init_rr,PREGRETS_rr])
np.savez(
    f"../results/PVAFM/GBT_pregrets_gamma0.9_oh_w_time0_2048_lookahead{6}_10reps.npz",
    t_list = Ts_offline,
    pregrets = PREGRETS_PVAFM_rr,
    )
PREGRETS_PVAFM_rr

[0, 0, np.int64(128), np.int64(256), np.int64(512), 750, 1024, 1250, 1500, 2048]


array([[2.21924298, 0.737035  , 0.0046656 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [2.29988117, 1.06555431, 0.52372975, 0.05190482, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [1.89185029, 1.21044262, 0.9262286 , 0.41591381, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [1.94902765, 1.58938239, 0.        , 0.11800085, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [2.05703421, 1.02055431, 0.50076721, 0.06024404, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [2.13830083, 1.18927797, 0.0074962 , 0.00466562, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [2.01267779, 1.11555431, 0.2703814 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [2.10138361, 1.11555431, 0.6051355 , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
